# Module 2, Section 3: Advanced Evaluation Techniques

In Sections 1 and 2, we used **final-answer evaluation** — scoring only the agent's last message against a ground truth answer. This is a great starting point, but it's a black box: it tells you _whether_ the agent got the right answer, not _how_ it got there.

As agents grow more complex, with multi-step reasoning, sub-agent routing, and HITL verification steps, final-answer eval alone leaves too many failure modes undetected. A response can be correct for the wrong reasons, or take a wildly inefficient path to get there.

In this section, we'll add three complementary evaluation techniques that give you visibility into the agent's internals:

| Eval Type | What it measures | Key question |
|---|---|---|
| **Final answer** _(Section 1)_ | Correctness of the last response | Did the agent give the right answer? |
| **Single-step** | One isolated decision point | Did the agent make the right choice at step X? |
| **Trajectory** | Full tool call sequence + execution path | Did the agent take the right path to the answer? |
| **Multi-turn simulation** | Full multi-message conversation | How does the agent behave across a realistic interaction? |

Each technique targets a different level of the agent's behavior. Together they form a comprehensive eval suite.

## Setup

In [ ]:
import uuid
from dotenv import load_dotenv

load_dotenv()

---
## 1. Single-Step Evaluation

Single-step evaluation isolates a **single decision point** in your agent and evaluates it independently — like a unit test for one node in your graph.

<div align="center">
    <img src="../../images/single-step-eval.png"  width="800">
</div>



**What we're testing:** The supervisor's routing decision. Given a customer question (with identity already verified), does the supervisor correctly route to `database_specialist` vs. `documentation_specialist`?

Wrong routing leads to the wrong tool being called, which leads to a wrong answer — but final-answer eval alone might not surface this if the agent recovers. Single-step eval catches the root cause directly.

**Why isolate?** By injecting a known `customer_id` directly into state, we bypass the HITL verification step entirely. This lets us test the routing logic in isolation — without noise from the upstream classifier.

### 1.1 Initialize the Supervisor Agent

In [ ]:
from agents.docs_agent import create_docs_agent
from agents.sql_agent import create_sql_agent
from agents.supervisor_agent import create_supervisor_agent
from agents.supervisor_hitl_agent import IntermediateState

sql_agent = create_sql_agent()
docs_agent = create_docs_agent()

# IntermediateState extends MessagesState with a `customer_id` field.
# We must pass it here so the supervisor's state schema includes `customer_id` —
# otherwise the value we inject at invoke time gets silently dropped and the
# dynamic prompt never receives it.
supervisor = create_supervisor_agent(
    db_agent=sql_agent,
    docs_agent=docs_agent,
    state_schema=IntermediateState,
)

### 1.2 Create Dataset

Our dataset contains questions paired with the expected routing target (`database_specialist` or `documentation_specialist`). Each example also includes a `customer_id` — passed into state to simulate an already-verified customer, keeping this a true unit test of routing logic only.

> **Note:** The supervisor tool names come from `supervisor_agent.py`: `"database_specialist"` wraps the SQL/DB agent, and `"documentation_specialist"` wraps the docs agent.

In [ ]:
from langsmith import Client

client = Client()

examples = [
    # Database questions — require database_specialist
    {
        "question": "What's the status of my most recent order?",
        "customer_id": "CUST-001",
        "route": "database_specialist",
    },
    {
        "question": "How much have I spent on monitors this year?",
        "customer_id": "CUST-007",
        "route": "database_specialist",
    },
    {
        "question": "How many orders did I place in the last 6 months?",
        "customer_id": "CUST-012",
        "route": "database_specialist",
    },
    # Documentation questions — require documentation_specialist
    {
        "question": "What's the return policy for opened electronics?",
        "customer_id": "CUST-001",
        "route": "documentation_specialist",
    },
    {
        "question": "Is the Sony WH-1000XM5 still under warranty?",
        "customer_id": "CUST-007",
        "route": "documentation_specialist",
    },
    {
        "question": "What accessories are compatible with TechHub laptops?",
        "customer_id": "CUST-012",
        "route": "documentation_specialist",
    },
]

dataset_name = f"techhub-single-step-routing-{uuid.uuid4()}"

dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Single-step eval: supervisor routing decisions (database_specialist vs documentation_specialist)",
)

client.create_examples(
    dataset_id=dataset.id,
    inputs=[
        {"question": ex["question"], "customer_id": ex["customer_id"]}
        for ex in examples
    ],
    outputs=[{"route": ex["route"]} for ex in examples],
)

print(f"Dataset: {dataset.url}")

### 1.3 Target Function

The target function invokes the supervisor with a pre-verified `customer_id` from the dataset, then extracts the **first tool call name** from the response — that's the routing decision.

We pass `interrupt_before=["tools"]` so execution stops immediately after the LLM selects a tool, before the sub-agent actually runs. This keeps the eval fast and truly isolated — we're testing the decision, not the downstream result.

In [ ]:
def run_supervisor_routing(inputs: dict) -> dict:
    """Invoke supervisor and capture the routing decision before any tool executes."""
    result = supervisor.invoke(
        {
            "messages": [{"role": "user", "content": inputs["question"]}],
            "customer_id": inputs["customer_id"],  # per-example, not hardcoded
        },
        config={"configurable": {"thread_id": str(uuid.uuid4())}},
        interrupt_before=["tools"],  # stop after the LLM decides, before sub-agent runs
    )
    # The last AIMessage with tool_calls contains the routing decision
    for msg in result["messages"]:
        if hasattr(msg, "tool_calls") and msg.tool_calls:
            return {"route": msg.tool_calls[0]["name"]}
    return {"route": None}

In [ ]:
# Quick test on one example
test_output = run_supervisor_routing(
    {"question": "What's the status of my recent order?", "customer_id": "CUST-001"}
)
print(f"Routing decision: {test_output['route']}")

### 1.4 Evaluator

A simple equality check — the routing decision should exactly match the expected sub-agent.

In [ ]:
def correct_route(outputs: dict, reference_outputs: dict) -> bool:
    """Check if the supervisor routed to the correct sub-agent."""
    return outputs["route"] == reference_outputs["route"]

### 1.5 Run Experiment

In [ ]:
results = client.evaluate(
    run_supervisor_routing,
    data=dataset_name,
    evaluators=[correct_route],
    experiment_prefix="supervisor-routing-single-step",
    description="Unit test: does the supervisor route to the correct sub-agent for each question type?",
    max_concurrency=3,
)

### 1.6 Analyze Results

Open the experiment link above and inspect any misrouted examples. A `correct_route = False` result tells you exactly which question caused the wrong routing decision — a much more targeted signal than a final-answer failure.

**Common failure patterns to look for:**
- Ambiguous questions (e.g., "tell me about my laptop") that could go to either sub-agent
- Questions that mention both account info and product info in one message
- Poorly worded questions where the intent is unclear

If you find failures, the fix is targeted: update the supervisor's system prompt rather than the entire pipeline.

---
## 2. Trajectory Evaluation

Trajectory evaluation examines the **complete execution path** of an agent run — which nodes were visited, which tools were called, and whether the agent followed the expected path to its answer.

<div align="center">
    <img src="../../images/trajectory-eval.png"  width="800">
</div>

A correct final answer doesn't guarantee an efficient or correct path. For example, an agent might:
- Call `execute_sql` three times when one JOIN query would suffice
- Skip the verification step for an account-specific query
- Route to the wrong sub-agent, get a bad result, then recover by trying the right one

**What we're testing:** The full `supervisor_hitl_agent` (with SQL agent) — including verification, routing, and tool call sequences.

**HITL handling:** Some questions require identity verification. The target function detects if the agent paused at an interrupt (waiting for email) and resumes with a per-example email from the dataset — no email baked into the question itself.

### 2.1 Initialize the Full HITL Agent

In [ ]:
from agents.supervisor_hitl_agent import create_supervisor_hitl_agent

# Use the SQL agent (from Section 2) with the full HITL pipeline
# use_checkpointer=True is required to call get_state() for HITL handling
agent = create_supervisor_hitl_agent(
    db_agent=create_sql_agent(),
    docs_agent=create_docs_agent(),
)

### 2.2 Create Dataset

Each example specifies:
- `question` — the customer's message
- `customer_email` — used to resume the HITL interrupt if triggered (not in the question)
- `expects_verification` — whether we expect the HITL verification step to run
- `expected_tools` — all tools we expect to be called, at every level: supervisor tools (`database_specialist`, `documentation_specialist`) **and** sub-agent tools (`execute_sql`, `search_policy_documents`, `search_product_documents`)

> **Why include sub-agent tools?** The sub-agents are invoked as Python functions inside tool wrappers, so their internal tool calls (e.g. `execute_sql`) don't appear in the parent graph's message state. To see them, we use **Run-based evaluators** that traverse the full LangSmith run tree via child runs — the same pattern used by `count_total_tool_calls_evaluator` in the `evaluators/` directory.

The last example is designed to stress-test the **efficiency** evaluator: a multi-part database question that may cause the SQL agent to make redundant `execute_sql` calls (querying the same data multiple ways) rather than consolidating into a single efficient query.

In [ ]:
from pprint import pprint

examples = [
    # Requires verification + DB query only
    {
        "question": "What were the items in my most recent order and what did each cost?",
        "customer_email": "sarah.chen@gmail.com",
        "expects_verification": True,
        "expected_tools": ["database_specialist", "execute_sql"],
    },
    # No verification needed + docs query only (return policy)
    {
        "question": "What's the return policy for opened electronics?",
        "customer_email": None,
        "expects_verification": False,
        "expected_tools": ["documentation_specialist", "search_policy_docs"],
    },
    # Requires verification + BOTH sub-agents called (cross-domain query)
    {
        "question": "Are any of my recent orders for items that are still within the return policy window?",
        "customer_email": "emily.rodriguez@icloud.com",
        "expects_verification": True,
        "expected_tools": [
            "database_specialist",
            "execute_sql",
            "documentation_specialist",
            "search_policy_docs",
        ],
    },
    # Requires verification + DB aggregation
    {
        "question": "How much have I spent on keyboards in total?",
        "customer_email": "james.nguyen@yahoo.com",
        "expects_verification": True,
        "expected_tools": ["database_specialist", "execute_sql"],
    },
    # No verification needed + product docs
    {
        "question": "What are the specs of the Sony WH-1000XM5?",
        "customer_email": None,
        "expects_verification": False,
        "expected_tools": ["documentation_specialist", "search_product_docs"],
    },
    # Efficiency stress test: complex multi-part DB question.
    # An efficient SQL agent should answer this in 1-2 queries.
    # A naive agent might call execute_sql separately for each part —
    # designed to trigger no_repeated_calls to fail and surface the redundancy.
    {
        "question": (
            "Give me a summary of my order history: how many orders have I placed, "
            "what's my total spend, and which product category do I buy most often?"
        ),
        "customer_email": "sarah.chen@gmail.com",
        "expects_verification": True,
        "expected_tools": ["database_specialist", "execute_sql"],
    },
]

pprint(examples[-1])

In [ ]:
trajectory_dataset_name = f"techhub-trajectory-eval-{uuid.uuid4()}"

trajectory_dataset = client.create_dataset(
    dataset_name=trajectory_dataset_name,
    description="Trajectory eval: expected tool call sequences for full HITL agent",
)

client.create_examples(
    dataset_id=trajectory_dataset.id,
    inputs=[
        {"question": ex["question"], "customer_email": ex["customer_email"]}
        for ex in examples
    ],
    outputs=[
        {
            "expects_verification": ex["expects_verification"],
            "expected_tools": ex["expected_tools"],
        }
        for ex in examples
    ],
)

print(f"Dataset: {trajectory_dataset.url}")

### 2.3 Target Function

The target function invokes the full agent and handles the HITL interrupt cleanly:

1. Run the agent with the customer's question
2. Check if the agent paused at an interrupt (identity verification needed)
3. If so, resume with the `customer_email` from the dataset — **no email in the question**
4. Return the complete message history for evaluation

In [ ]:
from langgraph.types import Command


def run_agent_trajectory(inputs: dict) -> dict:
    """Run the full HITL agent and return the complete message history."""
    thread_id = str(uuid.uuid4())
    config = {"configurable": {"thread_id": thread_id}}

    # Initial invocation
    agent.invoke(
        {"messages": [{"role": "user", "content": inputs["question"]}]},
        config=config,
    )

    # If the agent paused at the email verification interrupt, resume with the
    # customer email from the dataset (not embedded in the question).
    # Guard: only resume if customer_email is provided — calling Command(resume=None)
    # triggers a LangGraph bug. If state.next is True but no email is available,
    # the verification check will correctly fail (no "Verified!" message in state).
    state = agent.get_state(config)
    if state.next and inputs.get("customer_email"):
        agent.invoke(Command(resume=inputs["customer_email"]), config=config)

    # Return the full final message history
    final_state = agent.get_state(config)
    return {"messages": final_state.values["messages"]}

In [ ]:
# Quick test
test_result = run_agent_trajectory(
    {
        "question": "What's the return policy for opened electronics?",
        "customer_email": None,
    }
)

for msg in test_result["messages"]:
    msg.pretty_print()

### 2.4 Evaluators

We'll use four evaluators that together give a complete picture of trajectory quality:

1. **Verification step correct** — did the HITL verification node run when it should have?
2. **Expected tools called** — were all expected tools used, including sub-agent tools?
3. **No repeated calls** — did the agent avoid making redundant tool calls? (efficiency)
4. **[Optional] LLM-as-judge trajectory** — flexible quality assessment using `agentevals`

Evaluators 2 and 3 use the **`Run` object** — the LangSmith trace — rather than the message list. This gives access to the full run tree including child runs from sub-agents, so we can check `execute_sql` calls that never appear in the parent graph's message state.

The first evaluator is deterministic and cheap. The LLM judge catches nuanced issues the others can't.

In [ ]:
# Evaluator #1: Did the agent correctly route the question for identity verification?
def verification_step_correct(outputs: dict, reference_outputs: dict) -> dict:
    """Check whether the verify_customer node ran (or was correctly skipped).

    The verify_customer node adds an AIMessage containing 'Verified!' when it
    successfully validates a customer. We use this as a proxy to detect whether
    the HITL verification path was triggered.
    """
    messages = outputs["messages"]
    verified = any(
        "Verified!" in (getattr(msg, "content", "") or "") for msg in messages
    )
    return {
        "key": "verification_correct",
        "score": verified == reference_outputs["expects_verification"],
    }

In [ ]:
from collections import Counter

from langsmith.schemas import Run


# Helper
def extract_all_tool_names(run: Run) -> list[str]:
    """Recursively collect every tool call name from a run and all its child runs."""
    names = []
    if run.run_type == "tool":
        names.append(run.name)
    for child in run.child_runs or []:
        names.extend(extract_all_tool_names(child))
    return names


# Evaluator #2: Did the agent call all expected tools?
def expected_tools_called(run: Run, reference_outputs: dict) -> dict:
    """Check that all expected tools appear somewhere in the run tree.

    Because sub-agent tool calls (e.g. execute_sql) only appear as child runs in the
    LangSmith trace — not in the parent graph's message state — we traverse the full
    run tree via the Run object rather than inspecting messages.
    """
    actual = extract_all_tool_names(run)
    expected = reference_outputs["expected_tools"]
    missing = [t for t in expected if t not in actual]
    return {
        "key": "expected_tools_called",
        "score": len(missing) == 0,
        "comment": f"Missing: {missing}" if missing else "All expected tools called",
    }


# Evaluator #3: Did the agent avoid making redundant tool calls?
def no_repeated_calls(run: Run) -> dict:
    """Check that no tool was called more than once (efficiency guard).

    Repeated calls to the same tool — especially execute_sql — signal that the agent
    is making redundant queries instead of consolidating into a single efficient one.
    This connects directly to the SQL agent improvement from Section 2: the rigid
    db_agent often made multiple tool calls for questions a single SQL query could answer.
    """
    actual = extract_all_tool_names(run)
    counts = Counter(actual)
    repeated = {name: count for name, count in counts.items() if count > 1}
    return {
        "key": "no_repeated_calls",
        "score": len(repeated) == 0,
        "comment": str(repeated) if repeated else "No repeated calls",
    }

#### [Optional] LLM-as-Judge Trajectory Evaluator

The `agentevals` package provides an LLM-based trajectory evaluator that can assess overall path quality without needing a rigid reference. It's especially useful for catching issues the deterministic evaluators can't — like calling the right tool with a poorly formed query.

> **Trade-off:** Deterministic evaluators are fast, cheap, and consistent but require explicit ground truth. The LLM judge is flexible and catches nuanced issues, but is slower and less deterministic. In practice, use both.

In [ ]:
from agentevals.trajectory.llm import (
    TRAJECTORY_ACCURACY_PROMPT,
    create_trajectory_llm_as_judge,
)
from config import DEFAULT_MODEL

# Create the underlying agentevals judge
_trajectory_judge = create_trajectory_llm_as_judge(
    model=DEFAULT_MODEL,
    prompt=TRAJECTORY_ACCURACY_PROMPT,
)


def trajectory_accuracy(outputs: dict, **kwargs) -> dict:
    """Wrap agentevals LLM judge to extract messages from our outputs dict."""
    return _trajectory_judge(outputs=outputs["messages"])

### 2.5 Run Experiment

In [ ]:
trajectory_results = client.evaluate(
    run_agent_trajectory,
    data=trajectory_dataset_name,
    evaluators=[
        verification_step_correct,
        expected_tools_called,
        no_repeated_calls,
        trajectory_accuracy,
    ],
    experiment_prefix="full-agent-trajectory",
    description="Trajectory eval: verification step, tool call correctness, efficiency, and path quality",
    max_concurrency=2,
)

### 2.6 Analyze Results

In LangSmith, look at the four metrics together:

- **`verification_correct`** — if False, the HITL step was skipped when it shouldn't be (or triggered unnecessarily). This is a security-relevant failure.
- **`expected_tools_called`** — if False, the agent failed to gather the right information. Check the `comment` field to see which tools were missing. Because this evaluator traverses the full run tree, it catches missing sub-agent calls (e.g., `execute_sql` never ran).
- **`no_repeated_calls`** — if False, the agent made redundant tool calls. Check the `comment` field for which tools were repeated and how many times. This directly connects to the Section 2 story: the old `db_agent` would call a separate rigid tool for each piece of information, while the SQL agent should consolidate into fewer queries.
- **`trajectory_accuracy`** — the LLM judge's holistic verdict on path quality.

**Things to look for:**
- **Example 3** (cross-domain query): did the supervisor call *both* `database_specialist` and `documentation_specialist`? If only one was called, `expected_tools_called` will fail.
- **Example 6** (efficiency stress test): this is designed to reveal whether the SQL agent makes redundant `execute_sql` calls for a multi-part question. If `no_repeated_calls` fails here, the `comment` will show exactly which tool was called redundantly and how many times — giving you a concrete lead on how to improve the agent's SQL generation or prompt.

---
## 3. Multi-Turn Simulation

Single-step and trajectory eval test the agent with clean, pre-formatted inputs. Real customer interactions are messier — they span multiple turns, include ambiguity, and require the agent to handle the flow of a real conversation including the HITL verification step.

<div align="center">
    <img src="../../images/simulation-eval.png"  width="800">
</div>

**Multi-turn simulation** addresses this by replacing the human with an LLM-powered simulated user, generating a realistic conversation from scratch and evaluating the complete interaction.

**What we're testing:** Full conversations with the `supervisor_hitl_agent`, including:
- The agent asking for the customer's email when needed
- The customer providing their email naturally (in their own words)
- The agent retrieving the right information and responding helpfully
- Cases where verification should be skipped entirely

We'll use the [`openevals`](https://github.com/langchain-ai/openevals) library which provides the simulation infrastructure.

### 3.1 Build the App Wrapper

The `run_multiturn_simulation` function calls our app one message at a time. Each call receives a single message and should return a single response.

The key challenge: LangGraph's HITL uses `interrupt()` to pause the graph mid-run. When the agent is waiting for an email, the next user message should **resume** the interrupted graph — not start a new run.

We handle this by checking `agent.get_state(config).next` before each invocation:
- If the graph has a pending step (active interrupt), resume with `Command(resume=...)`
- Otherwise, invoke normally with the new message

In [ ]:
def make_app(agent):
    """Wrap a LangGraph HITL agent for use with run_multiturn_simulation.

    The simulator calls app(message, thread_id=...) once per turn.
    We detect active interrupts via get_state().next and route accordingly.
    """

    def app(inputs: dict, *, thread_id: str, **kwargs) -> dict:
        config = {"configurable": {"thread_id": thread_id}}
        state = agent.get_state(config)

        if state.next:  # graph is paused at an interrupt (waiting for email)
            result = agent.invoke(Command(resume=inputs["content"]), config=config)
        else:  # fresh turn — invoke normally
            result = agent.invoke({"messages": [inputs]}, config=config)

        return {"role": "assistant", "content": result["messages"][-1].content}

    return app

### 3.2 Create Dataset

Each example defines a **customer persona** for the simulated user and **success criteria** for the evaluator. The persona shapes how the simulated user behaves — when they provide their email, how frustrated they are, etc.

In [ ]:
simulation_examples = [
    {
        "simulated_user_prompt": (
            "You are a TechHub customer named Sarah Chen. Your email is sarah.chen@gmail.com. "
            "You want to find out exactly which items were in your most recent order and how much each item cost. "
            "Only provide your email address when the agent explicitly asks for it."
        ),
        "success_criteria": (
            "The agent asked for Sarah's email to verify her identity, successfully verified it, "
            "and provided a breakdown of the items and costs from her most recent order."
        ),
    },
    {
        "simulated_user_prompt": (
            "You are a frustrated TechHub customer. Your email is james.nguyen@yahoo.com. "
            "You placed an order about a week ago and it still hasn't shipped — you're annoyed and want to know what's going on. "
            "Start by asking about the order status without giving your email. "
            "When the agent asks for your email, provide it."
        ),
        "success_criteria": (
            "The agent verified James's identity, looked up the order status, "
            "and responded professionally and empathetically to his concern about the delayed shipment."
        ),
    },
    {
        "simulated_user_prompt": (
            "You are a TechHub customer who is considering buying a new laptop. "
            "You want to understand the return policy before purchasing. "
            "You have not made any purchases yet and prefer not to share personal information unless absolutely necessary."
        ),
        "success_criteria": (
            "The agent answered the return policy question clearly and directly without requesting "
            "identity verification, since no account information was needed."
        ),
    },
]

simulation_dataset_name = f"techhub-multiturn-simulation-{uuid.uuid4()}"

simulation_dataset = client.create_dataset(
    dataset_name=simulation_dataset_name,
    description="Multi-turn simulation: LLM-simulated customer conversations with HITL verification",
)

client.create_examples(
    dataset_id=simulation_dataset.id,
    inputs=[
        {"simulated_user_prompt": ex["simulated_user_prompt"]}
        for ex in simulation_examples
    ],
    outputs=[
        {"success_criteria": ex["success_criteria"]} for ex in simulation_examples
    ],
)

print(f"Dataset: {simulation_dataset.url}")

### 3.3 Target Function

The target function creates a fresh simulated user from the persona in the dataset, then runs the simulation. The returned `trajectory` is the full list of messages from the conversation.

In [ ]:
from openevals.simulators import create_llm_simulated_user, run_multiturn_simulation

# Re-create the agent instance for simulation (needs a fresh checkpointer state)
simulation_agent = create_supervisor_hitl_agent(
    db_agent=create_sql_agent(),
    docs_agent=create_docs_agent(),
)


def run_simulation(inputs: dict) -> dict:
    """Run a multi-turn simulation using the customer persona from the dataset."""
    user = create_llm_simulated_user(
        system=inputs["simulated_user_prompt"],
        model=DEFAULT_MODEL,
    )
    result = run_multiturn_simulation(
        app=make_app(simulation_agent),
        user=user,
        max_turns=6,
    )
    return {"trajectory": result["trajectory"]}

### 3.4 Evaluators

Multi-turn simulation outputs are inherently variable — each run produces a different conversation. LLM-as-judge evaluators are well-suited here because they can assess conversation quality holistically rather than checking for specific strings.

We'll use two evaluators:
- **`resolution`** — did the agent meet the success criteria defined in the dataset?
- **`num_turns`** — how many turns did it take? (lower = more efficient)

In [ ]:
from openevals.llm import create_llm_as_judge

resolution_evaluator = create_llm_as_judge(
    model=DEFAULT_MODEL,
    prompt=(
        "You are evaluating a customer support conversation.\n\n"
        "Success criteria:\n{reference_outputs}\n\n"
        "Conversation transcript:\n{outputs}\n\n"
        "Did the agent successfully meet the success criteria? "
        "Answer True if yes, False if not."
    ),
    feedback_key="resolution",
)


def num_turns(outputs: dict, **kwargs) -> dict:
    """Count the number of conversation turns (user + assistant pairs)."""
    return {"key": "num_turns", "score": len(outputs["trajectory"]) // 2}

### 3.5 Run Simulation

In [ ]:
simulation_results = client.evaluate(
    run_simulation,
    data=simulation_dataset_name,
    evaluators=[resolution_evaluator, num_turns],
    experiment_prefix="multiturn-simulation",
    description="Multi-turn simulation: LLM-simulated customer personas with HITL verification",
    max_concurrency=1,
)

### 3.6 Analyze Results

Open the experiment link above and review the simulation traces. Each trace shows the full simulated conversation — you can read the back-and-forth between the customer and agent.

**Key things to look for:**
- Did the agent ask for email verification when needed (example 1 & 2) and skip it when not needed (example 3)?
- How naturally did the agent handle the email request in context of a real conversation?
- Did `num_turns` stay reasonable (2–4 turns)? High turn counts may indicate the agent is asking unnecessary follow-up questions.
- For the frustrated customer (example 2): did the agent respond empathetically or robotically?

**Why simulation catches things trajectory eval misses:**  
Trajectory eval tests clean, pre-formatted inputs. Simulation tests with messy, realistic messages — "yeah it's james.nguyen@yahoo.com" is different from a clean extracted email, and the agent must handle both.

---
## Summary

You've now built a comprehensive evaluation suite for the TechHub agent:

| Section | Eval type | What it measures | Experiment prefix |
|---|---|---|---|
| Module 2, S1 | Final answer | Correctness of response | `baseline-eval` |
| Module 2, S2 | Final answer | Improvement from SQL agent | `with-sql-agent-eval` |
| **This section** | Single-step | Supervisor routing decision | `supervisor-routing-single-step` |
| **This section** | Trajectory | Tool call sequence + HITL path | `full-agent-trajectory` |
| **This section** | Simulation | Realistic multi-turn conversations | `multiturn-simulation` |

**When to use each:**
- **Final answer**: Validate correctness after any change. Catch regressions.
- **Single-step**: Debug a specific decision point. Fast feedback loop for targeted fixes.
- **Trajectory**: Catch inefficiencies, missing steps, or wrong paths before they surface in final answers.
- **Simulation**: Test realistic conversations including edge cases and multi-turn dynamics. Run before releasing to production.